# Final Project - CNN model
In this notebook we preprocess the car data and build a CNN classifier.

## Packages And Functions

In [ ]:
# Packages
from google.colab import drive
import os
from zipfile import ZipFile
from PIL import Image
import numpy as np
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from tensorflow.keras.layers import (Conv2D,
                          Dense,
                          Flatten,
                          AveragePooling2D,
                          Input,
                          Dropout,
                          BatchNormalization)
from keras.models import Sequential, Model
from keras.utils import to_categorical

# Document variables
IMAGE_SIZE = (128, 128)
RANDOM_STATE = 42

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Path to the zip file
zip_file_path = '/content/drive/MyDrive/Deep_Learning/data.zip'

# Extract images from the zip file
with ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall('/content/images')

# Path to the extracted images folder
images_folder_path = '/content/images/data'

# Load images with their labels
images = []
labels = []

for folder_name in os.listdir(images_folder_path):
    folder_path = os.path.join(images_folder_path, folder_name)
    if os.path.isdir(folder_path):
        for image_name in os.listdir(folder_path):
            image_path = os.path.join(folder_path, image_name)
            image = Image.open(image_path)
            image = image.resize(IMAGE_SIZE)
            image = np.array(image) / 255.0
            image = np.array(image)
            images.append(image)
            labels.append(folder_name)

# Convert lists to numpy arrays
images = np.array(images)
labels = np.array(labels)

## Load in the data

First we load in the data, containing images of cars from different sides. These are loaded in with their label into a dataframe

In [ ]:
# plot first image
plt.imshow(images[3005])
plt.title('Label: ' + labels[3005])
plt.axis('off')
plt.show()

In [ ]:
# Check if the data is balanced
unique, counts = np.unique(labels, return_counts=True)

np.asarray((unique, counts)).T

## Split the data
The data is normalized, one-hot encoded and split in train, validation and test data.

In [ ]:
X = images
# One-hot encoding
le = LabelEncoder().fit(labels)
y = le.transform(labels)
yy = to_categorical(y)

In [ ]:
# Data splitting
X_temp, X_test, y_temp, y_test = train_test_split(X, yy, test_size=0.2, random_state = RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state = RANDOM_STATE)

# Convert image data to TensorFlow tensors
X_train = tf.convert_to_tensor(X_train, dtype=tf.float32)
X_val = tf.convert_to_tensor(X_val, dtype=tf.float32)
X_test = tf.convert_to_tensor(X_test, dtype=tf.float32)

# Size data sets
print('Train set:', len(X_train))
print('Validation set:', len(X_val))
print('Test set:', len(X_test))

## CNN model
We will be constructing a CNN model to classify the images.
Architecture was derived from <a href="(https://www.learndatasci.com/tutorials/convolutional-neural-networks-image-classification">learndatasci.com</a>

In [ ]:
IMAGE_SIZE = (128,128) + (3,)
# The model

def CNN(input_shape=IMAGE_SIZE, num_classes=9):
    model = Sequential()
    model.add(Input(input_shape))
    model.add(Conv2D(64, kernel_size=(3,3), padding='same', activation='relu'))
    model.add(Conv2D(64, (3,3), activation='relu'))
    model.add(Dropout(0.2))
    model.add(Conv2D(32, (3,3), padding='same', activation='relu'))
    model.add(Conv2D(32, (3,3), activation='relu'))
    model.add(Dropout(0.2))
    model.add(AveragePooling2D((2,2)))
    model.add(Flatten())
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(16, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    return model

In [ ]:
# training the model
model = CNN()
history = model.fit(X_train, y_train, batch_size=32, epochs=10, validation_data=(X_val, y_val))

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model loss')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

## Testing model

In [ ]:
y_test_pred = model.predict(X_test)

In [ ]:
predictions = np.argmax(y_test_pred, axis=1)
true = np.argmax(y_test, axis=1)
print("Predictions")
print(predictions)
print("True labels")
print(true)

In [ ]:
print("Test Accuracy:", accuracy_score(true, predictions))
cf = confusion_matrix(true, predictions)
labeling = ["Driver side - Front",
            "Driver side - Rear",
            "Passenger side - Front",
            "Passenger side - Rear",
            "Driver side",
            "Front",
            "Passenger side",
            "Rear",
            "Unknown"]
sns.heatmap(cf, annot=True, yticklabels=labeling, xticklabels=labeling)
plt.xticks(rotation=45,  ha='right')
plt.show()